# Serve Unlimited-OCR on Colab GPU (T4)
Runtime -> Change runtime type -> **T4 GPU**, then Run All. Last cell prints the
URL for `UNLIMITED_OCR_URL=` in `~/.config/market-secrets/credentials.env`.

Serving uses **Transformers + Flask** (OpenAI-compatible shim), NOT vLLM:
debugged live 2026-07-29 — pip vLLM needs CUDA 13 (Colab has 12.x), fp16 trips a
masked_scatter_ dtype check (shimmed), fp32 OOMs both RAM and VRAM, and the
model's remote code needs transformers 4.46.x + addict and an output_path.

In [ ]:
!nvidia-smi -L
!pip -q install pymupdf flask addict easydict timm 'transformers==4.46.3' tokenizers accelerate

In [ ]:
# Write + start the OCR server (see markdown above for why this shape)
import base64, subprocess, time
open('/content/server.py','wb').write(base64.b64decode('aW1wb3J0IHRvcmNoCiMgVDQgbGFja3MgYmYxNiBhbmQgZnAxNiB0cmlwcyBtYXNrZWRfc2NhdHRlcl8gZHR5cGUgY2hlY2tzIGluIHRoZSB2aXNpb24gbWVyZ2U7CiMgc2hpbSBib3RoIHZhcmlhbnRzIHRvIGNhc3Qgc291cmNlIHRvIHRoZSBkZXN0aW5hdGlvbiBkdHlwZS4KX29tcyA9IHRvcmNoLlRlbnNvci5tYXNrZWRfc2NhdHRlcl8KZGVmIF9tcyhzZWxmLCBtYXNrLCBzb3VyY2UpOgogICAgcmV0dXJuIF9vbXMoc2VsZiwgbWFzaywgc291cmNlLnRvKHNlbGYuZHR5cGUpKQp0b3JjaC5UZW5zb3IubWFza2VkX3NjYXR0ZXJfID0gX21zCl9vbXNuID0gdG9yY2guVGVuc29yLm1hc2tlZF9zY2F0dGVyCmRlZiBfbXNuKHNlbGYsIG1hc2ssIHNvdXJjZSk6CiAgICByZXR1cm4gX29tc24oc2VsZiwgbWFzaywgc291cmNlLnRvKHNlbGYuZHR5cGUpKQp0b3JjaC5UZW5zb3IubWFza2VkX3NjYXR0ZXIgPSBfbXNuCgppbXBvcnQgYmFzZTY0LCBjb250ZXh0bGliLCBpbywgb3MsIHRlbXBmaWxlCmZyb20gZmxhc2sgaW1wb3J0IEZsYXNrLCByZXF1ZXN0LCBqc29uaWZ5CmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCk1PREVMID0gJ2JhaWR1L1VubGltaXRlZC1PQ1InCk9VVCA9ICcvY29udGVudC9vY3JvdXQnCm9zLm1ha2VkaXJzKE9VVCwgZXhpc3Rfb2s9VHJ1ZSkKdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoTU9ERUwsIHRydXN0X3JlbW90ZV9jb2RlPVRydWUpCm1vZGVsID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZChNT0RFTCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoX2R0eXBlPXRvcmNoLmZsb2F0MTYsIHVzZV9zYWZldGVuc29ycz1UcnVlKQptb2RlbCA9IG1vZGVsLmV2YWwoKS5jdWRhKCkKCmFwcCA9IEZsYXNrKF9fbmFtZV9fKQoKQGFwcC5nZXQoJy92MS9tb2RlbHMnKQpkZWYgbW9kZWxzKCk6CiAgICByZXR1cm4ganNvbmlmeSh7J29iamVjdCc6ICdsaXN0JywgJ2RhdGEnOiBbeydpZCc6IE1PREVMLCAnb2JqZWN0JzogJ21vZGVsJ31dfSkKCmRlZiBvY3JfaW1hZ2VfYnl0ZXMocG5nOiBieXRlcykgLT4gc3RyOgogICAgZiA9IHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZShzdWZmaXg9Jy5wbmcnLCBkZWxldGU9RmFsc2UpCiAgICBmLndyaXRlKHBuZyk7IGYuY2xvc2UoKQogICAgYnVmID0gaW8uU3RyaW5nSU8oKQogICAgd2l0aCBjb250ZXh0bGliLnJlZGlyZWN0X3N0ZG91dChidWYpOgogICAgICAgIG91dCA9IG1vZGVsLmluZmVyKHRvaywgcHJvbXB0PSc8aW1hZ2U+ZG9jdW1lbnQgcGFyc2luZy4nLAogICAgICAgICAgICAgICAgICAgICAgICAgIGltYWdlX2ZpbGU9Zi5uYW1lLCBvdXRwdXRfcGF0aD1PVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgaW1hZ2Vfc2l6ZT02NDAsIGNyb3BfbW9kZT1UcnVlKQogICAgb3MudW5saW5rKGYubmFtZSkKICAgIGlmIGlzaW5zdGFuY2Uob3V0LCBzdHIpIGFuZCBvdXQuc3RyaXAoKToKICAgICAgICByZXR1cm4gb3V0CiAgICB0eHQgPSBidWYuZ2V0dmFsdWUoKQogICAgaWYgbm90IHR4dC5zdHJpcCgpOgogICAgICAgIGltcG9ydCBnbG9iCiAgICAgICAgZnMgPSBbeCBmb3IgeCBpbiBzb3J0ZWQoZ2xvYi5nbG9iKE9VVCArICcvKicpLCBrZXk9b3MucGF0aC5nZXRtdGltZSkKICAgICAgICAgICAgICBpZiBvcy5wYXRoLmlzZmlsZSh4KV0KICAgICAgICBpZiBmczoKICAgICAgICAgICAgdHh0ID0gb3Blbihmc1stMV0sIGVycm9ycz0naWdub3JlJykucmVhZCgpCiAgICByZXR1cm4gdHh0CgpAYXBwLnBvc3QoJy92MS9jaGF0L2NvbXBsZXRpb25zJykKZGVmIGNoYXQoKToKICAgIGogPSByZXF1ZXN0LmdldF9qc29uKGZvcmNlPVRydWUpCiAgICB0ZXh0cyA9IFtdCiAgICBmb3IgbXNnIGluIGouZ2V0KCdtZXNzYWdlcycsIFtdKToKICAgICAgICBjb250ZW50ID0gbXNnLmdldCgnY29udGVudCcpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoY29udGVudCwgbGlzdCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIHAgaW4gY29udGVudDoKICAgICAgICAgICAgaWYgcC5nZXQoJ3R5cGUnKSA9PSAnaW1hZ2VfdXJsJzoKICAgICAgICAgICAgICAgIGI2NCA9IHBbJ2ltYWdlX3VybCddWyd1cmwnXS5zcGxpdCgnLCcsIDEpWzFdCiAgICAgICAgICAgICAgICB0ZXh0cy5hcHBlbmQob2NyX2ltYWdlX2J5dGVzKGJhc2U2NC5iNjRkZWNvZGUoYjY0KSkpCiAgICByZXR1cm4ganNvbmlmeSh7J29iamVjdCc6ICdjaGF0LmNvbXBsZXRpb24nLAogICAgICAgICAgICAgICAgICAgICdjaG9pY2VzJzogW3snaW5kZXgnOiAwLCAnZmluaXNoX3JlYXNvbic6ICdzdG9wJywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ21lc3NhZ2UnOiB7J3JvbGUnOiAnYXNzaXN0YW50JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2NvbnRlbnQnOiAnXG5cZlxuJy5qb2luKHRleHRzKX19XX0pCgphcHAucnVuKGhvc3Q9JzAuMC4wLjAnLCBwb3J0PTgwMDAsIHRocmVhZGVkPUZhbHNlKQo='))
sv = subprocess.Popen(['python3','/content/server.py'], stdout=open('/content/server.log','w'), stderr=subprocess.STDOUT)
exec("import requests\nok=False\nfor _ in range(60):\n    time.sleep(10)\n    if sv.poll() is not None:\n        print('server EXITED rc=', sv.returncode); print(open('/content/server.log').read()[-2000:]); break\n    try:\n        if requests.get('http://localhost:8000/v1/models', timeout=5).ok:\n            ok=True; print('SERVER IS UP'); break\n    except Exception:\n        pass\nprint('ok:', ok)")

In [ ]:
# Expose via Cloudflare quick tunnel. NOTE: tunnel caps requests at ~100s ->
# use UNLIMITED_OCR_PAGES_PER_CALL=1 on the client (~16s/page on T4).
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tun.stdout.readline())
    if m: url = m.group(0)
print('UNLIMITED_OCR_URL =', url)

Keep the tab open while using the endpoint; free tier disconnects after a few hours.